In [ ]:
# ahp层次分析法
import numpy as np

A = np.array([
    [1,   1/3, 1/2, 1/3],
    [3,   1,   2,   1],
    [2,   1/2, 1,   1/2],
    [3,   1,   2,   1]
], dtype=float)

# 求特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(A)

# 找最大特征值
max_index = np.argmax(eigenvalues.real)

lambda_max = eigenvalues[max_index].real

# 最大特征值对应的特征向量
w = eigenvectors[:, max_index].real

# 归一化
w = w / w.sum()

print("最大特征值：", lambda_max)
print("权重：", w)

最大特征值： 4.010362902240901
权重： [0.1091138  0.35091289 0.18906042 0.35091289]


In [4]:
# 一致性检验
import numpy as np

def ahp_weight(A):
    """
    AHP:
    1. 计算最大特征值
    2. 获取对应特征向量
    3. 归一化得到权重
    """

    A = np.array(A, dtype=float)

    # 特征值、特征向量
    eigenvalues, eigenvectors = np.linalg.eig(A)

    # 最大特征值
    index = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[index].real

    # 最大特征值对应特征向量
    w = eigenvectors[:, index].real

    # 防止出现整体负号
    if np.all(w < 0):
        w = -w

    # 归一化
    w = w / w.sum()

    return lambda_max, w


def consistency_ratio(A):
    """
    计算 AHP 一致性比例 CR
    """

    A = np.array(A, dtype=float)

    n = A.shape[0]

    lambda_max, _ = ahp_weight(A)

    # 一致性指标 CI
    CI = (lambda_max - n) / (n - 1)

    # RI
    RI_table = {
        1: 0,
        2: 0,
        3: 0.58,
        4: 0.90,
        5: 1.12,
        6: 1.24,
        7: 1.32,
        8: 1.41,
        9: 1.45,
        10: 1.49
    }

    RI = RI_table[n]

    if RI == 0:
        return 0

    CR = CI / RI

    return CR


A = np.array([
    [1,   1/3, 1/2, 1/3],
    [3,   1,   2,   1],
    [2,   1/2, 1,   1/2],
    [3,   1,   2,   1]
])

lambda_max, weight = ahp_weight(A)

print("最大特征值：", lambda_max)
print("权重：", weight)

CR = consistency_ratio(A)

print("一致性比例 CR：", CR)

if CR < 0.1:
    print("一致性检验通过")
else:
    print("一致性检验不通过，需要调整判断矩阵")

最大特征值： 4.010362902240901
权重： [0.1091138  0.35091289 0.18906042 0.35091289]
一致性比例 CR： 0.0038381119410745576
一致性检验通过


In [5]:
import numpy as np


# =========================
# 1. AHP 权重计算
# =========================

def ahp_weight(A):

    A = np.array(A, dtype=float)

    # 求特征值和特征向量
    eigenvalues, eigenvectors = np.linalg.eig(A)

    # 最大特征值
    index = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[index].real

    # 对应特征向量
    w = eigenvectors[:, index].real

    # 防止整体为负
    if np.all(w < 0):
        w = -w

    # 归一化
    w = w / w.sum()

    return lambda_max, w


# =========================
# 2. 一致性检验
# =========================

def check_consistency(A):

    A = np.array(A, dtype=float)

    n = A.shape[0]

    lambda_max, _ = ahp_weight(A)

    CI = (lambda_max - n) / (n - 1)

    RI_table = {
        1: 0,
        2: 0,
        3: 0.58,
        4: 0.90,
        5: 1.12,
        6: 1.24,
        7: 1.32,
        8: 1.41,
        9: 1.45,
        10: 1.49
    }

    RI = RI_table[n]

    if RI == 0:
        CR = 0
    else:
        CR = CI / RI

    return lambda_max, CI, CR


# =========================
# 3. 准则层判断矩阵
# =========================

criterion_matrix = np.array([
    [1,   1/3, 1/2, 1/3],
    [3,   1,   2,   1],
    [2,   1/2, 1,   1/2],
    [3,   1,   2,   1]
])


criterion_lambda, criterion_weight = ahp_weight(
    criterion_matrix
)

criterion_lambda, CI, CR = check_consistency(
    criterion_matrix
)

print("===== 准则层 =====")

print("权重：")
print(criterion_weight)

print("最大特征值：", criterion_lambda)
print("CI：", CI)
print("CR：", CR)

if CR < 0.1:
    print("一致性通过")
else:
    print("一致性不通过")


# =========================
# 4. 各指标下的方案权重
# =========================

# 花费
cost = np.array([1.0, 0.8, 1.5])
cost_matrix = cost[:, None] / cost[None, :]

# 景色
scenery = np.array([1.2, 1.0, 1.5])
scenery_matrix = scenery[:, None] / scenery[None, :]

# 交通
transport = np.array([1.2, 1.5, 1.0])
transport_matrix = transport[:, None] / transport[None, :]

# 安全
safety = np.array([1.0, 0.9, 1.2])
safety_matrix = safety[:, None] / safety[None, :]


# 分别求权重
_, cost_weight = ahp_weight(cost_matrix)
_, scenery_weight = ahp_weight(scenery_matrix)
_, transport_weight = ahp_weight(transport_matrix)
_, safety_weight = ahp_weight(safety_matrix)


# =========================
# 5. 构造局部权重矩阵
# =========================

local_weight = np.column_stack([
    cost_weight,
    scenery_weight,
    transport_weight,
    safety_weight
])

print("\n===== 局部权重矩阵 =====")

print(local_weight)


# =========================
# 6. 综合评价
# =========================

final_score = local_weight @ criterion_weight

print("\n===== 最终结果 =====")

cities = ["北京", "上海", "成都"]

for city, score in zip(cities, final_score):
    print(city, ":", score)


# =========================
# 7. 排序
# =========================

ranking = np.argsort(final_score)[::-1]

print("\n===== 排名 =====")

for i in ranking:
    print(cities[i], final_score[i])

===== 准则层 =====
权重：
[0.1091138  0.35091289 0.18906042 0.35091289]
最大特征值： 4.010362902240901
CI： 0.003454300746967102
CR： 0.0038381119410745576
一致性通过

===== 局部权重矩阵 =====
[[0.3030303  0.32432432 0.32432432 0.32258065]
 [0.24242424 0.27027027 0.40540541 0.29032258]
 [0.45454545 0.40540541 0.27027027 0.38709677]]

===== 最终结果 =====
北京 : 0.32138897324239235
上海 : 0.2998172045027388
成都 : 0.3787938222548688

===== 排名 =====
成都 0.3787938222548688
北京 0.32138897324239235
上海 0.2998172045027388


In [6]:
# 模糊评价分析
import numpy as np

# -------------------------
# 1. 指标权重
# -------------------------

A = np.array([
    0.30,
    0.20,
    0.35,
    0.15
])

# -------------------------
# 2. 模糊关系矩阵
# -------------------------

R = np.array([
    [0.05, 0.10, 0.25, 0.45, 0.15],
    [0.10, 0.20, 0.40, 0.25, 0.05],
    [0.05, 0.10, 0.20, 0.40, 0.25],
    [0.10, 0.20, 0.35, 0.25, 0.10]
])

# -------------------------
# 3. 模糊综合评价
# -------------------------

B = A @ R

print("综合评价向量：")
print(B)

# -------------------------
# 4. 最大隶属度原则
# -------------------------

levels = [
    "差",
    "较差",
    "一般",
    "较好",
    "好"
]

index = np.argmax(B)

print("最终评价等级：", levels[index])

# -------------------------
# 5. 量化评分
# -------------------------

scores = np.array([
    20,
    40,
    60,
    80,
    100
])

final_score = B @ scores

print("综合得分：", final_score)

综合评价向量：
[0.0675 0.135  0.2775 0.3625 0.1575]
最终评价等级： 较好
综合得分： 68.15


In [1]:
import numpy as np


def positive_normalize(x):
    """
    正向指标：越大越好
    """
    x = np.array(x, dtype=float)

    min_value = x.min()
    max_value = x.max()

    if max_value == min_value:
        return np.ones_like(x)

    return (x - min_value) / (max_value - min_value)


def negative_normalize(x):
    """
    负向指标：越小越好
    """
    x = np.array(x, dtype=float)

    min_value = x.min()
    max_value = x.max()

    if max_value == min_value:
        return np.ones_like(x)

    return (max_value - x) / (max_value - min_value)

x = np.array([20, 50, 90])

print(positive_normalize(x))

[0.         0.42857143 1.        ]


In [2]:
import numpy as np


X = np.array([
    [100, 20, 80],
    [150, 50, 50],
    [200, 90, 20]
], dtype=float)


# 第 1、2 列是正向指标
X_pos = X[:, [0, 1]]

# 第 3 列是负向指标
X_neg = X[:, [2]]


def positive_normalize(x):
    min_value = x.min(axis=0)
    max_value = x.max(axis=0)

    denominator = max_value - min_value

    return (x - min_value) / denominator


def negative_normalize(x):
    min_value = x.min(axis=0)
    max_value = x.max(axis=0)

    denominator = max_value - min_value

    return (max_value - x) / denominator


Z_pos = positive_normalize(X_pos)
Z_neg = negative_normalize(X_neg)

Z = np.hstack([Z_pos, Z_neg])

print(Z)

[[0.         0.         0.        ]
 [0.5        0.42857143 0.5       ]
 [1.         1.         1.        ]]
